> **Third-party integration.** Not maintained by BerriAI. SpendGuard is an open-source project ([github.com/m24927605/agentic-spendguard](https://github.com/m24927605/agentic-spendguard), Apache 2.0) authored by the contributor of this cookbook. Like the Arize, Langfuse, and Lunary cookbooks in this same directory, this notebook is provided by the integration vendor.

## Budget Guardrails — LiteLLM Proxy + SpendGuard

This notebook demonstrates how to use the LiteLLM Proxy with [SpendGuard](https://github.com/m24927605/agentic-spendguard) to gate every `/v1/chat/completions` call against a budget BEFORE the upstream provider is hit.

**What SpendGuard adds on top of LiteLLM's own spend tracking:**
- **Pre-call reserve** — SpendGuard's contract-DSL evaluator decides ALLOW / DENY / DEGRADE / REQUIRE_APPROVAL before LiteLLM forwards to the provider. Budget exhausted? HTTP 403, provider invoice clock never starts.
- **Post-call commit with reconciliation** — End-of-stream reconciler reads `response.usage.completion_tokens` and commits the real cost (not the worst-case estimator).
- **Signed append-only audit chain** — Every decision lands as a CloudEvent in `canonical_events`, with the 12-field LiteLLM-specific enrichment (`litellm_call_id`, `model`, `team_id`, `pricing_version`, etc) for forensics.
- **Fail-closed default** — Sidecar unreachable → HTTP 503; the request is denied, not forwarded.

SpendGuard wires in as a standard LiteLLM `CustomLogger` callback. No LiteLLM source change required.

## 1. Setup LiteLLM Proxy + SpendGuard

### 1.1 Install the SpendGuard SDK

```bash
pip install 'spendguard-sdk[litellm]==0.3.0'
```

The `litellm` extra pulls `litellm[proxy]` transitively so the proxy CLI works out of the box.

### 1.2 Run the SpendGuard sidecar

The SpendGuard sidecar is a Rust binary that runs alongside your LiteLLM proxy and exposes a Unix domain socket for the callback to talk to. The fastest path is the runnable example:

```bash
git clone https://github.com/m24927605/agentic-spendguard
cd agentic-spendguard/examples/litellm-proxy-composite
docker compose up --build
```

This brings up postgres + sidecar + your LiteLLM proxy in 3 containers.

### 1.3 Write your operator callback module

Save as `spendguard_callback.py` next to your `proxy_config.yaml`:

```python
import os
from spendguard._proto.spendguard.common.v1 import common_pb2
from spendguard.integrations.litellm import (
    BudgetBinding, ResolverContext, _LoopBoundCallback,
)

_UNIT = common_pb2.UnitRef(unit_id=os.environ['SPENDGUARD_UNIT_ID'],
                            token_kind='output_token', model_family='gpt-4')
_PRICING = common_pb2.PricingFreeze(
    pricing_version=os.environ['SPENDGUARD_PRICING_VERSION'],
    price_snapshot_hash=bytes.fromhex(os.environ['SPENDGUARD_PRICE_SNAPSHOT_HASH_HEX']),
    fx_rate_version=os.environ['SPENDGUARD_FX_RATE_VERSION'],
    unit_conversion_version=os.environ['SPENDGUARD_UNIT_CONVERSION_VERSION'],
)
_BINDING = BudgetBinding(
    budget_id=os.environ['SPENDGUARD_BUDGET_ID'],
    window_instance_id=os.environ['SPENDGUARD_WINDOW_INSTANCE_ID'],
    unit=_UNIT, pricing=_PRICING,
)

def _resolve(ctx: ResolverContext) -> BudgetBinding | None:
    # In production: dispatch by ctx.user_api_key_dict.team_id
    return _BINDING

def _estimate(ctx):
    # `amount_atomic` is the worst-case pre-call cost expressed in
    # the BudgetBinding's unit. With `unit=output_token` above,
    # '50' means "reserve 50 output tokens upfront". Production code
    # should derive this from your pricing table — e.g. estimated
    # max response tokens for the model. The reservation is
    # reconciled to the real `usage.completion_tokens` in
    # `_reconcile` below, and any overshoot is refunded.
    return [common_pb2.BudgetClaim(
        budget_id=_BINDING.budget_id, unit=_UNIT, amount_atomic='50',
        direction=common_pb2.BudgetClaim.DEBIT,
        window_instance_id=_BINDING.window_instance_id,
    )]

def _reconcile(ctx, response):
    tokens = int(getattr(getattr(response, 'usage', None), 'completion_tokens', 0) or 0)
    return [common_pb2.BudgetClaim(
        budget_id=_BINDING.budget_id, unit=_UNIT, amount_atomic=str(max(tokens, 1)),
        direction=common_pb2.BudgetClaim.DEBIT,
        window_instance_id=_BINDING.window_instance_id,
    )]

handler_instance = _LoopBoundCallback(
    socket_path=os.environ['SPENDGUARD_SIDECAR_UDS'],
    tenant_id=os.environ['SPENDGUARD_TENANT_ID'],
    budget_resolver=_resolve,
    claim_estimator=_estimate,
    claim_reconciler=_reconcile,
)
```

### 1.4 Update proxy_config.yaml

```yaml
model_list:
  - model_name: gpt-4o-mini
    litellm_params:
      model: openai/gpt-4o-mini
      api_key: os.environ/OPENAI_API_KEY

litellm_settings:
  callbacks: ["spendguard_callback.handler_instance"]
  drop_params: true

general_settings:
  master_key: os.environ/LITELLM_MASTER_KEY
```

Set `PYTHONPATH` to include the directory of `spendguard_callback.py` before launching the proxy.

### 1.5 Launch the proxy

```bash
export SPENDGUARD_SIDECAR_UDS=/var/run/spendguard/adapter.sock
export SPENDGUARD_TENANT_ID=...
export SPENDGUARD_BUDGET_ID=...
# ... (full env-var list at github.com/m24927605/agentic-spendguard/blob/main/docs/specs/litellm-integration/PROXY_RECIPE.md)

python -m litellm.proxy.proxy_cli --config proxy_config.yaml --port 4000
```

## 2. Make LLM Requests to the Gated Proxy

Standard OpenAI client — no SpendGuard SDK in your app code. The gating is invisible until the budget hits a limit.

In [ ]:
import openai

client = openai.OpenAI(
    base_url="http://localhost:4000",
    api_key="sk-demo-key",  # LITELLM_MASTER_KEY
)

# ALLOW path — SpendGuard reserves + commits normally.
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "hello"}],
)
print(response.choices[0].message.content)

## 3. Verify the Audit Chain

Every call writes a signed CloudEvent to `canonical_events`. Cross-table join with LiteLLM's own `LiteLLM_SpendLogs` (if you have a SpendLogs database configured):

```sql
SELECT
  ce.event_type,
  ce.tenant_id,
  cost_advisor_safe_decode_payload(ce.payload_json)->'spendguard'->>'litellm_call_id'
    AS litellm_call_id,
  cost_advisor_safe_decode_payload(ce.payload_json)->'spendguard'->>'model' AS model,
  cost_advisor_safe_decode_payload(ce.payload_json)->'spendguard'->>'team_id' AS team_id,
  cost_advisor_safe_decode_payload(ce.payload_json)->>'final_decision' AS decision
FROM canonical_events ce
WHERE ce.event_type IN ('spendguard.audit.decision', 'spendguard.audit.outcome')
  AND ce.event_time > NOW() - INTERVAL '1 hour'
ORDER BY ce.event_time DESC;
```

The `spendguard` JSONB sub-object on each row carries the 12-field LiteLLM-specific enrichment per [DESIGN §8.2a](https://github.com/m24927605/agentic-spendguard/blob/main/docs/specs/litellm-integration/DESIGN.md).

## 4. Going Further

- **Multi-tenant via virtual keys:** the operator template above is single-team. To dispatch per LiteLLM team, inspect `ctx.user_api_key_dict.team_id` in `_resolve` and look up the binding in your control plane. Full template: [PROXY_RECIPE.md §2](https://github.com/m24927605/agentic-spendguard/blob/main/docs/specs/litellm-integration/PROXY_RECIPE.md).
- **Streaming:** SpendGuard's `_async_log_success_streaming` reconciles end-of-stream `usage.completion_tokens` automatically. No app code change.
- **Direct (non-proxy) async callers:** `from spendguard.integrations.litellm import SpendGuardDirectAcompletion` wraps `litellm.acompletion()`. Sync `litellm.completion()` is not supported; route via the SpendGuard egress proxy for that.
- **Fail-open dev override:** `SPENDGUARD_LITELLM_FAIL_OPEN=1` lets calls through when the sidecar is unreachable (development only).

Runnable end-to-end demo:
```bash
git clone https://github.com/m24927605/agentic-spendguard
cd agentic-spendguard
make demo-up DEMO_MODE=litellm_real   # 4-step ALLOW + DENY + STREAM + MULTI-TEAM
make demo-up DEMO_MODE=litellm_deny   # 3 fail-closed sub-steps
```